# Loading and handling data

[Download notebook](notebooks/07_handling_data.ipynb).

In this chapter we learn how to load data from various sources and how
to handle tabular data using `pandas`. Before we start, we describe what
a dataframe is, since it is the building block of many data science
routines.

## Dataframes (`pandas`)

`pandas` is the standard library for handling tabular data in Python.
Its central data structure is the `DataFrame`, which is a
two-dimensional, tabular data structure — essentially a table with rows
and columns. Each column has a name and a data type (e.g., integers,
floats, strings), and each row represents one observation or record. You
can think of a dataframe as an in-memory spreadsheet that comes with
powerful methods for selecting, filtering, transforming, and aggregating
data. In Python, the `pandas` library provides the `DataFrame` class,
which is the standard tool for working with tabular data. We will see
how to create and manipulate dataframes below.

In [1]:
# Execute once for installing pandas
# !pip3 install pandas
import pandas as pd
import numpy as np

Here are the most important routines, if `df` is a `DataFrame`:

- `pd.DataFrame(dict)`: create a DataFrame from a dictionary (keys
  become column names).
- `df["col"]`: select a single column (returns a `Series`).
- `df[["col1", "col2"]]`: select multiple columns.
- `df.iloc[i]`: select row `i` by position.
- `df.loc[i, "col"]`: select by label.
- `df[df["col"] > value]`: filter rows by a condition.
- `df.head(n)` / `df.tail(n)`: first / last `n` rows.
- `df.describe()`: summary statistics (count, mean, std, min, quartiles,
  max) for all numeric columns.
- `df.sort_values("col")`: sort by a column.
- `df.groupby("col").mean()`: group by a column and compute means.
- `df.isna()`: boolean mask of missing values.
- `df.dropna()`: drop rows with missing values.
- `df.fillna(value)`: fill missing values.
- `df.drop(columns=["col"])`: remove a column.
- `df.to_numpy()`: convert to a NumPy array.

In [2]:
df = pd.DataFrame({
    "age": [22, 27, 31, 45, 38, 29],
    "income": [42000, 51000, 64000, 83000, 72000, 56000],
    "hours_studied": [10, 12, 7, 4, 6, 9],
    "passed": [1, 1, 0, 0, 1, 1]
})
df

In [3]:
# selecting columns
print(df["age"])

0    22
1    27
2    31
3    45
4    38
5    29
Name: age, dtype: int64

In [4]:
print(df[["age", "income"]])

   age  income
0   22   42000
1   27   51000
2   31   64000
3   45   83000
4   38   72000
5   29   56000

In [5]:
# selecting rows
print(df.iloc[0])

age                 22
income           42000
hours_studied       10
passed               1
Name: 0, dtype: int64

In [6]:
print(df.loc[0:2, ["age", "passed"]])

   age  passed
0   22       1
1   27       1
2   31       0

In [7]:
# filtering
filtered = df[df["income"] > 55000]
filtered

In [8]:
# summary statistics
print(df.describe())

             age        income  hours_studied    passed
count   6.000000      6.000000       6.000000  6.000000
mean   32.000000  61333.333333       8.000000  0.666667
std     8.246211  14827.901627       2.898275  0.516398
min    22.000000  42000.000000       4.000000  0.000000
25%    27.500000  52250.000000       6.250000  0.250000
50%    30.000000  60000.000000       8.000000  1.000000
75%    36.250000  70000.000000       9.750000  1.000000
max    45.000000  83000.000000      12.000000  1.000000

In [9]:
# sorting
df_sorted = df.sort_values("income", ascending=False)
print(df_sorted)

   age  income  hours_studied  passed
3   45   83000              4       0
4   38   72000              6       1
2   31   64000              7       0
5   29   56000              9       1
1   27   51000             12       1
0   22   42000             10       1

In [10]:
# groupby
grouped = df.groupby("passed").mean(numeric_only=True)
print(grouped)

         age   income  hours_studied
passed                              
0       38.0  73500.0           5.50
1       29.0  55250.0           9.25

In [11]:
# missing values
df_missing = df.copy()
df_missing.loc[2, "income"] = np.nan
df_missing.loc[4, "hours_studied"] = np.nan

print(df_missing)
print()
print(df_missing.isna().sum())
print()
print(df_missing.dropna())

   age   income  hours_studied  passed
0   22  42000.0           10.0       1
1   27  51000.0           12.0       1
2   31      NaN            7.0       0
3   45  83000.0            4.0       0
4   38  72000.0            NaN       1
5   29  56000.0            9.0       1

age              0
income           1
hours_studied    1
passed           0
dtype: int64

   age   income  hours_studied  passed
0   22  42000.0           10.0       1
1   27  51000.0           12.0       1
3   45  83000.0            4.0       0
5   29  56000.0            9.0       1

In [12]:
# fill missing values with column means
df_filled = df_missing.fillna(df_missing.mean(numeric_only=True))
print(df_filled)

   age   income  hours_studied  passed
0   22  42000.0           10.0       1
1   27  51000.0           12.0       1
2   31  60800.0            7.0       0
3   45  83000.0            4.0       0
4   38  72000.0            8.4       1
5   29  56000.0            9.0       1

In [13]:
# from pandas to numpy
clean = df_missing.dropna()
X = clean[["age", "income", "hours_studied"]].to_numpy(dtype=float)
y = clean["passed"].to_numpy(dtype=float)
print("X.shape =", X.shape)
print("y =", y)

X.shape = (4, 3)
y = [1. 1. 0. 1.]

## Adding and renaming columns (`assign`, `rename`)

Adding new columns and renaming existing ones are among the most
frequent pandas operations.

- `df["new_col"] = values`: add a new column (or overwrite an existing
  one) by direct assignment.
- `df.assign(col=expr)`: return a new DataFrame with an additional
  column (does not modify the original).
- `df.rename(columns={"old": "new"})`: rename one or more columns.
- `df.insert(loc, column, value)`: insert a column at a specific
  position.

In [14]:
df = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie"],
    "weight_kg": [62, 85, 74],
    "height_m": [1.65, 1.80, 1.75]
})
df

In [15]:
# adding a computed column
df["bmi"] = df["weight_kg"] / df["height_m"] ** 2
df

In [16]:
# assign returns a new DataFrame, leaving df unchanged
df2 = df.assign(height_cm=df["height_m"] * 100)
df2

In [17]:
# renaming columns
df_renamed = df.rename(columns={"weight_kg": "weight", "height_m": "height"})
df_renamed

Note that direct assignment (`df["col"] = ...`) modifies the DataFrame
in place, while `assign` and `rename` return new DataFrames by default.

## Merging and joining DataFrames (`merge`, `concat`)

Combining data from multiple tables is a core data-handling task.
`pd.merge` joins two DataFrames on a common key (like an SQL JOIN),
while `pd.concat` stacks them vertically or horizontally.

- `pd.merge(left, right, on="key")`: inner join on a shared column.
- `pd.merge(left, right, on="key", how="left")`: left join (keep all
  rows from the left DataFrame). Other options: `"right"`, `"outer"`.
- `pd.concat([df1, df2])`: stack DataFrames vertically (row-wise).
- `pd.concat([df1, df2], axis=1)`: stack DataFrames horizontally
  (column-wise).

In [18]:
students = pd.DataFrame({
    "student_id": [1, 2, 3, 4],
    "name": ["Alice", "Bob", "Charlie", "Diana"]
})

grades = pd.DataFrame({
    "student_id": [1, 2, 3, 5],
    "grade": [1.3, 2.0, 1.7, 2.7]
})

print(students)
print()
print(grades)

   student_id     name
0           1    Alice
1           2      Bob
2           3  Charlie
3           4    Diana

   student_id  grade
0           1    1.3
1           2    2.0
2           3    1.7
3           5    2.7

In [19]:
# inner join: only students present in both tables
merged = pd.merge(students, grades, on="student_id")
print(merged)

   student_id     name  grade
0           1    Alice    1.3
1           2      Bob    2.0
2           3  Charlie    1.7

In [20]:
# left join: keep all students, NaN where no grade exists
merged_left = pd.merge(students, grades, on="student_id", how="left")
print(merged_left)

   student_id     name  grade
0           1    Alice    1.3
1           2      Bob    2.0
2           3  Charlie    1.7
3           4    Diana    NaN

In [21]:
# concatenating vertically
df_a = pd.DataFrame({"name": ["Alice", "Bob"], "score": [85, 92]})
df_b = pd.DataFrame({"name": ["Charlie", "Diana"], "score": [78, 91]})
combined = pd.concat([df_a, df_b], ignore_index=True)
print(combined)

      name  score
0    Alice     85
1      Bob     92
2  Charlie     78
3    Diana     91

When concatenating vertically, use `ignore_index=True` to reset the
index; otherwise both DataFrames keep their original indices, which may
lead to duplicate index values.

## Applying custom functions (`apply`, `map`)

Built-in methods like `.mean()` or `.sum()` cover many cases, but
sometimes you need to apply an arbitrary Python function to each row,
column, or element.

- `df.apply(func)`: apply `func` to each column (default) or each row
  (`axis=1`).
- `series.map(func)`: apply `func` element-wise to a Series.
- `df.map(func)`: apply `func` element-wise to every cell of a
  DataFrame.

In [22]:
df = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie"],
    "score": [85, 92, 78],
    "bonus": [5, 3, 8]
})
df

In [23]:
# apply a function to each column (default axis=0)
print(df[["score", "bonus"]].apply(np.mean))

score    85.000000
bonus     5.333333
dtype: float64

In [24]:
# apply a function row-wise (axis=1)
df["total"] = df.apply(lambda row: row["score"] + row["bonus"], axis=1)
df

In [25]:
# map: element-wise transformation of a Series
df["grade"] = df["total"].map(lambda x: "pass" if x >= 85 else "fail")
df

Use `apply` with `axis=1` when the computation depends on multiple
columns in the same row. For simple element-wise transformations on a
single column, `map` is more readable. For best performance on large
DataFrames, prefer vectorized operations (e.g.,
`df["score"] + df["bonus"]`) over `apply` whenever possible.

## Value counts and unique values (`value_counts`, `unique`)

When exploring a dataset, you often want to know which values occur in a
column and how frequently. This is especially useful for categorical
data.

- `df["col"].value_counts()`: count how often each value appears, sorted
  by frequency.
- `df["col"].unique()`: return an array of all distinct values.
- `df["col"].nunique()`: return the number of distinct values.
- `df["col"].value_counts(normalize=True)`: return relative frequencies
  instead of absolute counts.

In [26]:
df = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Diana", "Eve", "Frank"],
    "department": ["Sales", "IT", "Sales", "IT", "HR", "Sales"],
    "level": ["junior", "senior", "senior", "junior", "junior", "senior"]
})
df

In [27]:
# how often does each department appear?
print(df["department"].value_counts())

department
Sales    3
IT       2
HR       1
Name: count, dtype: int64

In [28]:
# relative frequencies
print(df["department"].value_counts(normalize=True))

department
Sales    0.500000
IT       0.333333
HR       0.166667
Name: proportion, dtype: float64

In [29]:
# unique values and their count
print("Unique departments:", df["department"].unique())
print("Number of departments:", df["department"].nunique())

Unique departments: <StringArray>
['Sales', 'IT', 'HR']
Length: 3, dtype: str
Number of departments: 3

`value_counts()` is one of the first things to try when getting to know
a new dataset. For numeric columns, consider using
`df["col"].describe()` instead.

## Handling duplicates (`duplicated`, `drop_duplicates`)

Real-world data often contains duplicate rows — for example from
repeated data imports or logging errors. `pandas` makes it easy to
detect and remove them.

- `df.duplicated()`: return a boolean Series that is `True` for
  duplicate rows.
- `df.duplicated(subset=["col1", "col2"])`: check for duplicates based
  on specific columns only.
- `df.drop_duplicates()`: remove duplicate rows (keeps the first
  occurrence by default).
- `df.drop_duplicates(subset=["col"], keep="last")`: keep the last
  occurrence instead.

In [30]:
df = pd.DataFrame({
    "name": ["Alice", "Bob", "Alice", "Charlie", "Bob"],
    "score": [85, 92, 85, 78, 95]
})
df

In [31]:
# which rows are duplicates?
print(df.duplicated())

0    False
1    False
2     True
3    False
4    False
dtype: bool

In [32]:
# remove exact duplicate rows
print(df.drop_duplicates())

      name  score
0    Alice     85
1      Bob     92
3  Charlie     78
4      Bob     95

In [33]:
# duplicates based on a subset of columns
print(df.duplicated(subset=["name"]))

0    False
1    False
2     True
3    False
4     True
dtype: bool

In [34]:
# keep only the last entry per name
print(df.drop_duplicates(subset=["name"], keep="last"))

      name  score
2    Alice     85
3  Charlie     78
4      Bob     95

Note that `drop_duplicates()` returns a new DataFrame by default. To
modify in place, pass `inplace=True`. When checking duplicates on a
subset of columns, the first (or last) complete row is kept, not just
the key columns.

## Loading data from files (`csv`, `xlsx`, `json`)

Data often comes as files. The most common formats are CSV
(comma-separated values), Excel (`.xlsx`), and JSON (JavaScript Object
Notation). A CSV file is like an Excel table, where contents of cells
are separated by a comma (or some other delimiter). A JSON file has the
same structure as a Python dictionary. We will see an example below.

- `pd.read_csv(path_or_buffer)`: reads a CSV file (or string via
  `io.StringIO`) into a DataFrame.
- `df.to_csv(path, index=False)`: writes a DataFrame to a CSV file.
- `pd.read_json(path_or_buffer)`: reads a JSON file (or string) into a
  DataFrame.
- `df.to_json()`: converts a DataFrame to a JSON string.
- `pd.read_excel(path)`: reads an Excel file into a DataFrame (requires
  the `openpyxl` package).

In [35]:
import io

# We simulate a CSV file using a string
csv_string = """name,age,city
Alice,30,Berlin
Bob,25,Munich
Charlie,35,Hamburg"""

df = pd.read_csv(io.StringIO(csv_string))
print(df)

      name  age     city
0    Alice   30   Berlin
1      Bob   25   Munich
2  Charlie   35  Hamburg

In [36]:
# Reading JSON data into a DataFrame
json_string = '[{"name": "Alice", "age": 30, "city": "Berlin"}, {"name": "Bob", "age": 25, "city": "Munich"}]'
df = pd.read_json(io.StringIO(json_string))
print(df)

    name  age    city
0  Alice   30  Berlin
1    Bob   25  Munich

In [37]:
# Converting a DataFrame back to JSON
print(df.to_json(orient="records", indent=2))

[
  {
    "name":"Alice",
    "age":30,
    "city":"Berlin"
  },
  {
    "name":"Bob",
    "age":25,
    "city":"Munich"
  }
]

For Excel files, `pd.read_excel("data.xlsx")` requires the `openpyxl`
package. We will not go into any more detail here.

## Loading data from online resources (`requests`)

The `requests` library allows downloading data from the internet.

- `requests.get(url)`: sends a GET request (i.e. tries to download a
  website) and returns a response object.
- `response.status_code`: HTTP status code (200 means success).
- `response.text`: the response body as a string.
- `response.json()`: parse the response body as JSON.

In [38]:
import requests

url = "https://jsonplaceholder.typicode.com/todos/1"
response = requests.get(url)
print("status code:", response.status_code)
print("content:", response.json())

status code: 200
content: {'userId': 1, 'id': 1, 'title': 'delectus aut autem', 'completed': False}

You can also combine `requests` with `pandas`:
`pd.read_csv("https://example.com/data.csv")` reads a CSV file directly
from a URL.

As a concrete example, the REST Countries API provides geographic and
demographic data. Here we fetch population data for all countries and
display the ten most populous ones.

In [39]:
response = requests.get("https://restcountries.com/v3.1/all?fields=name,population,region")
countries = response.json()

df_pop = pd.DataFrame([{
    "country": c["name"]["common"],
    "population": c["population"],
    "region": c["region"]
} for c in countries])

df_pop.nlargest(10, "population")

In [40]:
# population by region
df_pop.groupby("region")["population"].sum().sort_values(ascending=False)

region
Asia         4724731966
Africa       1462464411
Americas     1042579783
Europe        741657922
Oceania        48059678
Antarctic          1700
Name: population, dtype: int64

## Working with dates and times (`to_datetime`, `Timestamp`)

`pandas` can parse and work with dates.

- `pd.to_datetime(...)`: convert strings to datetime.
- `dt.strftime(format)`: format a datetime as a string.

In [41]:
dates = pd.to_datetime(["2024-01-15", "2024-03-22", "2024-07-01"])
print(dates)
print("Month:", dates.month.tolist())
print("Day of week:", dates.day_of_week.tolist())

DatetimeIndex(['2024-01-15', '2024-03-22', '2024-07-01'], dtype='datetime64[us]', freq=None)
Month: [1, 3, 7]
Day of week: [0, 4, 0]

In [42]:
dt = pd.Timestamp("2024-06-15 14:30:00")
print(dt.strftime("%-m/%-d/%y, %H:%M"))

6/15/24, 14:30

## SQL databases (`sqlite3`)

Python has built-in support for SQLite databases via the `sqlite3`
module. This is useful for working with structured data that lives in a
database.

- `sqlite3.connect(path)`: connect to a database (use `":memory:"` for
  an in-memory database).
- `cursor.execute(sql)`: execute an SQL statement.
- `cursor.fetchall()`: fetch all results.
- `pd.read_sql_query(sql, conn)`: run an SQL query and return the result
  as a DataFrame.

In [43]:
import sqlite3

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE students (
        name TEXT,
        age INTEGER,
        grade REAL
    )
""")

students = [
    ("Alice", 22, 1.3),
    ("Bob", 25, 2.0),
    ("Charlie", 23, 1.7),
    ("Diana", 27, 2.3),
]

cursor.executemany("INSERT INTO students VALUES (?, ?, ?)", students)
conn.commit()

In [44]:
# querying
cursor.execute("SELECT * FROM students WHERE age > 23")
for row in cursor.fetchall():
    print(row)

('Bob', 25, 2.0)
('Diana', 27, 2.3)

In [45]:
# reading SQL results into a pandas DataFrame
df_sql = pd.read_sql_query("SELECT * FROM students", conn)
print(df_sql)
conn.close()

      name  age  grade
0    Alice   22    1.3
1      Bob   25    2.0
2  Charlie   23    1.7
3    Diana   27    2.3